# High-dimensional covariatesBehaviour as the covariate dimension `p` grows with the sample size held fixed.## About this notebookThe model, loss, sampler, metrics and data generation all live in the`rnn_agt` package. This notebook sets up an experiment and reports it; nothingis redefined here, so every notebook and driver in the repository shares oneimplementation.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
P_VALUES = [10, 30, 50, 100, 300, 500, 1000]N_TRAIN, N_TEST = 1000, 2000CENSORING = 0.25def signal(X):    """Sparse signal: only the first three covariates carry information.    The remaining p-3 columns are noise, so this measures how well the model    resists dimensions that do not matter.    """    return D.f_interaction(X)rows = []for p in P_VALUES:    seeds = make_seeds(31 + p)    rng = seeds.data()    tau = D.calibrate_tau(N_TRAIN, signal, "normal", rng, CENSORING,                          D.DEPENDENCE_SPECS["ar1"])    tr = D.make_dataset(N_TRAIN, "interaction", "normal", rng,                        dependence="ar1", tau=tau, p=p)    te = D.make_dataset(N_TEST, "interaction", "normal", rng,                        dependence="ar1", tau=tau, p=p)    cfg = TrainConfig(model="rnn_agt", epochs=10, pair_sample_s=30,                      hidden_dim=64, gru_layers=2, lr=3e-4)    res = train_model(tr, te, p, cfg, make_seeds(11))    rows.append({        "p": p,        "train C": res.metrics["train_cindex"], "test C": res.metrics["test_cindex"],        "train AMSE": res.metrics["train_amse"], "test AMSE": res.metrics["test_amse"],        "params": res.n_params,    })    print(f"p={p:5d}  params={res.n_params:8,d}  "          f"test C={res.metrics['test_cindex']:.3f}  "          f"test AMSE={res.metrics['test_amse']:.2f}", flush=True)highdim = pd.DataFrame(rows)highdim.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))axes[0].plot(highdim.p, highdim["train C"], marker="o", label="train")axes[0].plot(highdim.p, highdim["test C"], marker="s", label="test")axes[0].axhline(0.5, ls="--", c="grey", lw=1)axes[0].set_ylabel("IPCW C-index")axes[1].plot(highdim.p, highdim["train AMSE"], marker="o", label="train")axes[1].plot(highdim.p, highdim["test AMSE"], marker="s", label="test")axes[1].set_ylabel("AMSE")for ax in axes:    ax.set_xscale("log"); ax.set_xlabel("covariate dimension p")    ax.legend(); ax.grid(alpha=.3)fig.suptitle(f"High-dimensional behaviour (n_train={N_TRAIN}, only 3 informative covariates)")fig.tight_layout()fig.savefig("highdim_results.png", dpi=150)plt.show()print("The train/test gap is the quantity of interest: it widens as p grows "      "because the network has more noise dimensions to fit. Report both "      "curves, not test alone.")